In [1]:
import requests
import pandas as pd
from pathlib import Path

# Cairo coordinates
LATITUDE = 30.0444
LONGITUDE = 31.2357

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "pressure_msl",
        "wind_speed_10m",
        "wind_direction_10m",
        "cloud_cover"
    ],

    "timezone": "Africa/Cairo",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm"
}

response = requests.get(URL, params=params)
response.raise_for_status()

data = response.json()

df = pd.DataFrame(data["hourly"])

df.rename(columns={
    "time": "datetime",
    "temperature_2m": "temperature_c",
    "relative_humidity_2m": "humidity_percent",
    "precipitation": "precipitation_mm",
    "rain": "rain_mm",
    "pressure_msl": "pressure_hpa",
    "wind_speed_10m": "wind_speed_kmh",
    "wind_direction_10m": "wind_direction",
    "cloud_cover": "cloud_cover_percent"
}, inplace=True)

df["latitude"] = LATITUDE
df["longitude"] = LONGITUDE
df["location"] = "Cairo"

output = Path("data/weather/weather_cairo.csv")
output.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} rows")
print(output)

Saved 17544 rows
data\weather\weather_cairo.csv


In [2]:
import os
from pathlib import Path
import pandas as pd
import requests

# 1. Configuration & Target Directory
LATITUDE = 30.0444
LONGITUDE = 31.2357
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

target_dir = r"D:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed"
os.makedirs(target_dir, exist_ok=True)

URL = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "pressure_msl",
        "wind_speed_10m",
        "wind_direction_10m",
        "cloud_cover",
    ],
    "timezone": "Africa/Cairo",
    "temperature_unit": "celsius",
    "wind_speed_unit": "kmh",
    "precipitation_unit": "mm",
}

print("📥 Fetching Cairo Weather Data from Open-Meteo API...")
response = requests.get(URL, params=params)
response.raise_for_status()
data = response.json()

# 2. Convert to DataFrame & Rename Columns
df = pd.DataFrame(data["hourly"])
df.rename(
    columns={
        "time": "datetime",
        "temperature_2m": "temperature_c",
        "relative_humidity_2m": "humidity_percent",
        "precipitation": "precipitation_mm",
        "rain": "rain_mm",
        "pressure_msl": "pressure_hpa",
        "wind_speed_10m": "wind_speed_kmh",
        "wind_direction_10m": "wind_direction",
        "cloud_cover": "cloud_cover_percent",
    },
    inplace=True,
)

# 3. Feature Engineering (Adding Derived Columns)
df["datetime"] = pd.to_datetime(df["datetime"])
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
df["day_of_week"] = df["datetime"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([4, 5]).astype(int)  # Fri & Sat

# Weather condition flags
df["is_rainy"] = (df["precipitation_mm"] > 0).astype(int)
df["extreme_heat"] = (df["temperature_c"] >= 38.0).astype(int)

# Save Raw Hourly Weather Data to processed directory
raw_weather_path = os.path.join(target_dir, "weather_cairo.csv")
df.to_csv(raw_weather_path, index=False, encoding="utf-8-sig")
print(f"✅ Saved Raw Weather Data: {len(df):,} rows -> {raw_weather_path}")

# 4. Create Hourly Weather Profile (Aggregated by Hour of Day 0-23)
hourly_profile = (
    df.groupby("hour")
    .agg(
        avg_temp_c=("temperature_c", "mean"),
        max_temp_c=("temperature_c", "max"),
        avg_humidity_percent=("humidity_percent", "mean"),
        avg_wind_speed_kmh=("wind_speed_kmh", "mean"),
        rain_probability=("is_rainy", "mean"),
    )
    .reset_index()
    .round(2)
)

# Save Aggregated Hourly Profile to processed directory
profile_path = os.path.join(target_dir, "weather_hourly_profile.csv")
hourly_profile.to_csv(profile_path, index=False)
print(
    f"✅ Saved Aggregated Weather Profile: {len(hourly_profile)} hourly records -> {profile_path}"
)

# Print Preview
print("\n📊 Hourly Weather Profile Preview (First 5 Hours):")
print(hourly_profile.head())

📥 Fetching Cairo Weather Data from Open-Meteo API...
✅ Saved Raw Weather Data: 17,544 rows -> D:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\weather_cairo.csv
✅ Saved Aggregated Weather Profile: 24 hourly records -> D:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed\weather_hourly_profile.csv

📊 Hourly Weather Profile Preview (First 5 Hours):
   hour  avg_temp_c  max_temp_c  avg_humidity_percent  avg_wind_speed_kmh  \
0     0       21.79        33.1                 54.57                9.45   
1     1       20.95        32.4                 59.06                8.52   
2     2       20.18        31.4                 63.42                7.87   
3     3       19.50        30.6                 67.36                7.29   
4     4       19.08        29.4                 69.99                6.84   

   rain_probability  
0              0.01  
1              0.00  
2              0.00  
3              0.00  
4              0.00  


In [3]:
import os
import sqlite3
import numpy as np
import pandas as pd

# 1. تحديد مسار القراءة (processed) ومسار الحفظ النهائي (data)
processed_dir = r"D:\Courses\MachineLearning\Depi\Final Project\data\transportation\processed"
final_output_dir = r"D:\Courses\MachineLearning\Depi\Final Project\data"

os.makedirs(final_output_dir, exist_ok=True)

print("⚙️ Building Master Dataset and SQL Database...")

# 2. قراءة البيانات المجهزة من processed
stop_features = pd.read_csv(os.path.join(processed_dir, "stop_features.csv"))
weather_profile = pd.read_csv(
    os.path.join(processed_dir, "weather_hourly_profile.csv")
)

# 3. حساب مؤشرات الطقس الأساسية
city_weather_summary = {
    "overall_avg_temp_c": weather_profile["avg_temp_c"].mean().round(2),
    "overall_max_temp_c": weather_profile["max_temp_c"].max().round(2),
    "overall_avg_humidity": weather_profile["avg_humidity_percent"]
    .mean()
    .round(2),
    "overall_rain_prob": weather_profile["rain_probability"].mean().round(3),
}

# 4. دمج البيانات ومعالجة الأعمدة
master_df = stop_features.copy()
for key, value in city_weather_summary.items():
    master_df[key] = value

master_df["avg_headway_min"] = (master_df["avg_headway_secs"] / 60).round(1)
master_df["district_name"] = master_df["district_name"].fillna("Unknown")

# One-Hot Encoding لاسم الحي
master_df = pd.get_dummies(
    master_df,
    columns=["district_name"],
    prefix="district",
    drop_first=False,
    dtype=int,
)

# 5. حفظ الملف النهائي master_dataset.csv في مجلد data الرئيسي
final_csv_path = os.path.join(final_output_dir, "master_dataset.csv")
master_df.to_csv(final_csv_path, index=False)

# 6. تصدير قاعدة بيانات SQLite (urbanmind_ai.db) في مجلد data الرئيسي أيضاً
final_db_path = os.path.join(final_output_dir, "urbanmind_ai.db")
conn = sqlite3.connect(final_db_path)

tables = [
    "stops",
    "routes",
    "trips",
    "stop_times",
    "frequencies",
    "integrated_stops",
    "weather_hourly_profile",
]
for table_name in tables:
    file_path = os.path.join(processed_dir, f"{table_name}.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)

master_df.to_sql("master_dataset", conn, if_exists="replace", index=False)
conn.close()

print("\n" + "=" * 60)
print("🎉 FINAL MASTER DATASET & DATABASE SAVED SUCCESSFULLY!")
print("=" * 60)
print(f"📁 Final CSV Location: {final_csv_path}")
print(f"📁 Final DB Location:  {final_db_path}")
print(
    f"• Total Rows: {len(master_df):,} | Features: {len(master_df.columns)}"
)
print("=" * 60)

⚙️ Building Master Dataset and SQL Database...

🎉 FINAL MASTER DATASET & DATABASE SAVED SUCCESSFULLY!
📁 Final CSV Location: D:\Courses\MachineLearning\Depi\Final Project\data\master_dataset.csv
📁 Final DB Location:  D:\Courses\MachineLearning\Depi\Final Project\data\urbanmind_ai.db
• Total Rows: 2,468 | Features: 15
